# unit03 レッスン: 勾配ブースティング木を主砲にする

**昨日(unit02)やったこと** — このコンペの正しい検証は **`GroupKFold(5)` を `groups=train["product_key"]` で切る**こと、
そして `list_price` / `discount_rate` は目的変数から逆算された**リーク列**なので特徴量に入れてはいけないこと。
ここまでが確定した。**今日はその CV の上にモデルを載せる。**

**データは昨日と同じ**(`unit02-validation-and-leakage/data/`)。3日かけて同じコンペを改善していく:

| Day | やること | 状態 |
|---|---|---|
| Day2 (unit02) | 信じられる CV を作る | **済**(GroupKFold + リーク列の除外) |
| **Day3 (このユニット)** | **その上にモデルを載せる** | ← いまここ |
| Day4 (unit04) | 特徴量を足してスコアを押し上げる | この先 |

## このレッスンを終えると作れるようになるもの

1. scikit-learn の **estimator API**(`fit` / `predict` / `get_params` / `set_params`)という共通インターフェースを理解し、
   **CV ループのコードを1行も変えずにモデルだけ差し替えて**比較できる
2. 決定木 → ランダムフォレスト → 勾配ブースティングの違いを説明でき、
   **どんなデータで木が強く、どんなデータで線形モデルが強いか**を実測で判断できる
3. **前処理が要るモデルと要らないモデル**を区別し、線形モデルには `SimpleImputer` + `StandardScaler` を繋げられる
4. LightGBM を **early stopping** 込みで回し、fold ごとの `best_iteration_` を取り出して全データ再学習に使える
5. `feature_importances_` の **split と gain** の違いを説明でき、**重要度トップの列をまずリークで疑える**

所要の目安: **90〜120分**。このあと演習 `ex01`〜`ex04` が続く。

## このレッスンの読み方

セルは**上から順に**実行する。構成は概念ごとに次の8ステップの繰り返し:

| 記号 | 内容 |
|---|---|
| ① | なぜこれを学ぶのか(実務のどこで使うか) |
| ② | 解説(C# との対応表・API 表) |
| ③ | **見る** — 完成コードを実行して結果を見る |
| ④ | **予測する** — 次のセルの結果を、実行する前に予想する |
| ⑤ | **変えてみる** — ③ の条件を変えて実行し、予測と照合する |
| ⑥ | **書いてみる**(指示) |
| ⑦ | **書いてみる**(君が書くセル) |
| ⑧ | チェックポイント(即時採点) |

⑦ を飛ばしても後続セルは動く(⑧ が `[NG]` を出すだけ)。詰まったら ⑤ に戻ればよい。

In [ ]:
# ===== セットアップ: このセルを最初に1回だけ実行する =====
import os

# 小さいデータでは、スレッドを増やすほど並列化のオーバーヘッドが勝って**遅くなる**。
# 1スレッドに固定すると速く、しかも結果が完全に再現する(本番の大きなデータでは外してよい)。
os.environ.setdefault("OMP_NUM_THREADS", "1")

import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import lightgbm as lgb
from lightgbm import LGBMRegressor

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 30)

# データは unit02 と同じものを使う(このコンペを Day2→Day3→Day4 と3日かけて改善していく)。
# notebook をこのユニット直下で開いても、リポジトリのルートで開いても動くようにする。
DATA = Path("../unit02-validation-and-leakage/data")
if not (DATA / "train.csv").exists():
    DATA = Path("courses/kaggle-sprint/unit02-validation-and-leakage/data")
assert (DATA / "train.csv").exists(), f"train.csv が見つかりません: {DATA.resolve()}"

print("pandas:", pd.__version__, "/ numpy:", np.__version__,
      "/ scikit-learn:", sklearn.__version__, "/ lightgbm:", lgb.__version__)
print("DATA =", DATA.resolve())

train = pd.read_csv(DATA / "train.csv", parse_dates=["collected_at"])
print("train:", train.shape)


# ---------- 採点ヘルパー(中身は読まなくてよい) ----------
def check(name, actual, expected, hint=""):
    import numpy as _np
    try:
        ok = actual is not None and bool(_np.all(_np.isclose(_np.asarray(actual, dtype=float), _np.asarray(expected, dtype=float))))
    except (TypeError, ValueError):
        ok = actual == expected
    if ok:
        print(f"[OK] {name}: 正解!")
    else:
        print(f"[NG] {name}: 期待値 {expected!r} / 実際 {actual!r}")
        if hint:
            print(f"     ヒント: {hint}")
    return ok


def check_frame(name, actual, shape=None, columns=None, hint=""):
    """DataFrame の形と列名を採点する。DataFrame でなくても例外にしない。"""
    if not isinstance(actual, pd.DataFrame):
        print(f"[NG] {name}: 期待値 pandas.DataFrame(shape={shape}, columns={columns}) / 実際 {type(actual).__name__}")
        if hint:
            print(f"     ヒント: {hint}")
        return False
    problems = []
    if shape is not None and tuple(actual.shape) != tuple(shape):
        problems.append(f"shape の期待値 {tuple(shape)} / 実際 {tuple(actual.shape)}")
    if columns is not None and list(actual.columns) != list(columns):
        problems.append(f"列名の期待値 {list(columns)} / 実際 {list(actual.columns)}")
    if problems:
        print(f"[NG] {name}: " + " | ".join(problems))
        if hint:
            print(f"     ヒント: {hint}")
        return False
    print(f"[OK] {name}: 正解!")
    return True


def call_safely(fn, *args, **kwargs):
    """未完成の関数を呼んでも notebook が止まらないようにするラッパ。例外なら None を返す。"""
    if not callable(fn):
        return None
    try:
        return fn(*args, **kwargs)
    except Exception as e:
        print(f"     (関数の中で例外が出ました → {type(e).__name__}: {e})")
        return None


def param_of(model, key):
    """model.get_params()[key] を安全に取り出す。取れなければ None。"""
    try:
        return model.get_params().get(key)
    except Exception:
        return None


def col_of(df, col):
    """df[col] を list で安全に取り出す。取れなければ None。"""
    if not isinstance(df, pd.DataFrame) or col not in df.columns:
        return None
    return list(df[col])


print("\nセットアップ完了。ヘルパー: check / check_frame / call_safely / param_of / col_of")

---
# 概念1 — sklearn の estimator API という「型」

## ① なぜ: モデルは「部品」であって「主役」ではない

コンペでも実務でも、1つのデータに対して試すモデルは1つではない。
「まず定数、次に線形、次に GBDT、余裕があればニューラルネット」と**片っ端から回して CV で選ぶ**のが普通の進め方だ。
このとき毎回、データの分割・OOF の組み立て・スコアの計算を**モデルごとに書き直していたら**、
比較の条件がズレて「どちらが良いのか」が分からなくなる(そしてバグる)。

scikit-learn の世界では、これが起きない。**すべてのモデルが同じインターフェースを実装している**からだ。
昨日書いた `GroupKFold` のループは、**中の `model` を差し替えるだけ**でそのまま使い回せる。
今日の主砲である LightGBM も、この共通インターフェースに乗って提供されている(`LGBMRegressor`)。

だから最初に学ぶのはモデルの中身ではなく、**「型」のほう**だ。ここを押さえると、
このあと出てくる知らないモデルも「同じ形をした部品」として扱えるようになる。

## ② 解説: `IEstimator` を実装したクラス群を DI で差し替える、そのもの

### C# との対応表

| C# での構図 | scikit-learn |
|---|---|
| `interface IEstimator { IEstimator Fit(double[][] x, double[] y); double[] Predict(double[][] x); }` | すべてのモデルが `fit(X, y)` / `predict(X)` を持つ |
| 具象クラス `RidgeEstimator : IEstimator`, `LgbmEstimator : IEstimator` | `Ridge()`, `LGBMRegressor()`, `DummyRegressor()` |
| 呼び出し側は**インターフェース型の変数**で受ける | `model = 何でもよい` → `model.fit(...)` |
| DI コンテナで実装を差し替える | `for model in [Ridge(), LGBMRegressor()]:` |
| コンストラクタ引数 = 設定値 | ハイパーパラメータ(`Ridge(alpha=1.0)`) |
| 設定を後から読む/変える | `get_params()` / `set_params(**kw)` |
| プロトタイプの複製(設定だけコピー) | `sklearn.base.clone(model)` |
| 未初期化フィールドを触ると例外 | `fit` 前に `coef_` を触ると `AttributeError` |

**「呼び出し側は具象型を知らない」**という一点で、C# の DI とまったく同じ設計だ。
`fit` が**自分自身を返す**(C# の Fluent API と同じ)ので、`model.fit(X, y).predict(X)` と繋げて書ける。

### 覚えるべき規約は2つだけ

1. **`fit(X, y)` で学習し、`predict(X)` で予測する。** 引数の形は `X` が `(n_samples, n_features)` の2次元、`y` が `(n_samples,)` の1次元。
   pandas の `DataFrame` / `Series` をそのまま渡せる(内部で NumPy 配列に変換される)。
2. **学習後にだけ存在する属性は、末尾にアンダースコアが付く。**
   `coef_`(線形モデルの係数)、`feature_importances_`(木の重要度)、`best_iteration_`(early stopping で選ばれた木の本数)、
   `constant_`(定数モデルが返す値)。これは「`fit` していない状態で触ったらバグ」という**規約による型安全**だと思えばよい。

### 今日いきなり使う3つのモデル

| モデル | 何をするものか | 主な設定 |
|---|---|---|
| `DummyRegressor(strategy="mean")` | **常に同じ値**(学習データの平均や中央値)を返す。「モデルの形をした定数」。unit01 の定数提出の estimator 版で、**これを下回るモデルには存在価値がない**という下限 | `strategy="mean" / "median"` |
| `Ridge(alpha=...)` | 線形回帰(`y = w·x + b`)に、**係数の二乗和のペナルティ**(L2正則化)を足したもの。`alpha` が大きいほど係数を0に近づけ、慎重な(単純な)モデルになる | `alpha` |
| `LGBMRegressor(...)` | 勾配ブースティング決定木。中身は概念2〜3でやる。**sklearn 互換**なので同じ `fit`/`predict` で動く | 概念3の表 |

### API 一覧

| 用途 | API | 戻り値 | 注意 |
|---|---|---|---|
| 学習 | `model.fit(X, y)` | **自分自身**(`self`) | だから `.fit(X, y).predict(X)` と繋げられる |
| 予測 | `model.predict(X)` | `(n_samples,)` の `ndarray` | 入力が DataFrame でも返りは NumPy 配列 |
| 設定の取得 | `model.get_params()` | `dict` | キーはコンストラクタ引数名と同じ |
| 設定の変更 | `model.set_params(**kw)` | **自分自身** | 学習済みの状態は捨てられる |
| 設定だけ複製 | `sklearn.base.clone(model)` | 同じ設定の**未学習**の新インスタンス | 学習済みの重みはコピー**されない**。CV の各 fold で「まっさらな同じ設定」を作るのに使う |

> **なぜ `clone` が要るのか** — CV のループで同じインスタンスを使い回して `fit` すると、
> 「前の fold の学習結果がリセットされているか」がモデル実装依存になる。`clone` を挟めば**必ず未学習から**始まる。
> C# で言えば「シングルトンではなく transient で解決する」に相当する。

In [ ]:
# GOAL: 昨日の GroupKFold ループの中に estimator を1つ置くだけで、OOF 予測とスコアが出ることを見る

from sklearn.base import clone
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold

# STEP 1: 特徴量行列 X と目的変数 y を作る。まず shape を必ず print する。
#   使わない列: record_id(ID) / product_key(グループキー) / model_code(商品をほぼ一意に指す = 使うと丸暗記)
#                list_price, discount_rate(unit02 で確定したリーク列) / price(目的変数そのもの)
train["days"] = (train["collected_at"] - train["collected_at"].min()).dt.days   # 起点からの経過日数
FEAT_NUM = ["brand_tier", "views", "title_len", "days"]
FEAT_CAT = ["category", "site", "condition"]

# pd.get_dummies(df, columns=[...]) — カテゴリ列を 0/1 の列に開く(one-hot)。
#   線形モデルは文字列を食べられないので、まずは全部数値にするのが最短。
#   エンコーディングの体系的な話は unit04 でやる。
X = pd.get_dummies(train[FEAT_NUM + FEAT_CAT], columns=FEAT_CAT)
y = np.log1p(train["price"].to_numpy(dtype=float))   # unit02 で確定: 対数空間で学習・評価(= RMSLE)
groups = train["product_key"]                        # unit02 で確定: 同一商品は同じ fold に閉じ込める

print("X:", X.shape, " y:", y.shape, " groups:", groups.shape)
print("X の列:", list(X.columns))
print("X の dtype 内訳:", {str(k): int(v) for k, v in X.dtypes.value_counts().to_dict().items()})

# STEP 2: fit する前に学習後の属性を触ると何が起きるか(規約の確認)
model = DummyRegressor(strategy="mean")
try:
    model.constant_
except AttributeError as e:
    print("\nfit 前に constant_ を触ると:", type(e).__name__)

# STEP 3: GroupKFold の 5 fold を回して OOF(out-of-fold)予測を作る。中身は昨日と同じループ。
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(y))
for k, (idx_tr, idx_va) in enumerate(gkf.split(X, y, groups)):
    m = clone(model)                       # 毎 fold まっさらな同じ設定のモデルを作る
    m.fit(X.iloc[idx_tr], y[idx_tr])       # 学習
    oof[idx_va] = m.predict(X.iloc[idx_va])  # 検証 fold の行だけを埋める
    print(f"  fold{k}: train={X.iloc[idx_tr].shape} valid={X.iloc[idx_va].shape} "
          f"予測値={m.constant_.ravel()[0]:.4f}")

# STEP 4: OOF が全行ちょうど1回ずつ埋まっているか(昨日の規律)
print("\noof:", oof.shape, " 未記入の行数:", int((oof == 0).sum()))
rmse = float(np.sqrt(np.mean((oof - y) ** 2)))
print(f"DummyRegressor(mean) の CV RMSE(= log 空間なので RMSLE): {rmse:.6f}")

# STEP 5: fit は自分自身を返すので繋げて書ける(C# の Fluent API と同じ)
pred_all = DummyRegressor(strategy="mean").fit(X, y).predict(X)
print("繋げて書いた予測:", pred_all.shape, "先頭3件:", pred_all[:3].round(4))

## ④ 予測: ループの中の1行を差し替えると、何が変わる?

次のセルでは、上のループを `cv_rmse(model)` という関数に包んで、**モデルだけ**を差し替える。
ループのコードは1文字も変えない。

| モデル | 中身 |
|---|---|
| `DummyRegressor(strategy="mean")` | 常に平均を返す(= ③ で実測した `1.111857`) |
| `Ridge(alpha=1.0)` | 線形回帰 + L2正則化 |
| `LGBMRegressor(...)`(木300本) | 今日の主砲 |

実行する前に予測しよう。

1. **3つのスコアの順位**はどうなる? (RMSE は小さいほど良い)
2. `Ridge` と `LGBMRegressor` の差は、どのくらい開くと思う?
3. `Ridge(alpha=1.0)` を `Ridge(alpha=1000.0)` にしたら、スコアは良くなる? 悪くなる?
   (`alpha` は「係数を0に近づける力」だったことを思い出そう)
4. `model.get_params()` を print したら、`LGBMRegressor` のキーはいくつくらいあると思う?

> ヒント: このデータの `price` がどう作られていたかを思い出すと、順位は予想できる。
> unit02 のデータ説明では「カテゴリごとの基準価格 + ブランドティア + 状態 + サイト + 時間による下落」を
> **足し合わせて** `exp` を取っていた。つまり **log を取った空間では、ほぼ足し算**だ。

In [ ]:
# GOAL: CV ループを関数に包み、estimator だけを差し替えて比較する(ループのコードは共通)

from sklearn.linear_model import Ridge


def cv_rmse(model, X=X, y=y, groups=groups, n_splits=5, return_oof=False):
    """GroupKFold で OOF 予測を作り、log 空間の RMSE を返す。
    model は fit / predict を持つものなら何でもよい(= estimator API に乗っていれば何でも)。"""
    gkf = GroupKFold(n_splits=n_splits)
    oof = np.zeros(len(y))
    for idx_tr, idx_va in gkf.split(X, y, groups):
        m = clone(model)
        m.fit(X.iloc[idx_tr], y[idx_tr])
        oof[idx_va] = m.predict(X.iloc[idx_va])
    score = float(np.sqrt(np.mean((oof - y) ** 2)))
    return (score, oof) if return_oof else score


# STEP 1: LightGBM の設定を1か所にまとめておく(中身は概念3で説明する)
LGB_BASE = dict(n_estimators=300, learning_rate=0.05, num_leaves=31, min_child_samples=20,
                subsample=0.9, subsample_freq=1, colsample_bytree=0.9, reg_lambda=1.0,
                random_state=42, n_jobs=1, verbose=-1)

# STEP 2: 同じ関数に、違う estimator を渡すだけ
candidates = [
    ("DummyRegressor(mean)", DummyRegressor(strategy="mean")),
    ("Ridge(alpha=1.0)", Ridge(alpha=1.0)),
    ("Ridge(alpha=1000.0)", Ridge(alpha=1000.0)),
    ("LGBMRegressor(300本)", LGBMRegressor(**LGB_BASE)),
]
print(f"{'モデル':<26}{'CV RMSE':>10}")
print("-" * 36)
for name, m in candidates:
    print(f"{name:<22}{cv_rmse(m):>10.6f}")

# STEP 3: get_params は「設定の辞書」。set_params で後から変えられる
r = Ridge(alpha=1.0)
print("\nRidge の設定:", r.get_params())
print("LGBMRegressor の設定キー数:", len(LGBMRegressor().get_params()))

r.set_params(alpha=42.0)            # 自分自身を返すので繋げても書ける
print("set_params 後の alpha:", r.get_params()["alpha"])

# STEP 4: clone は「設定だけ複製した未学習のモデル」。元のモデルは壊れない
fitted = Ridge(alpha=1.0).fit(X, y)
print("\nfit 済みモデルの coef_:", fitted.coef_.shape, " intercept_:", round(float(fitted.intercept_), 4))
copy_ = clone(fitted)
print("clone した側に coef_ はあるか:", hasattr(copy_, "coef_"), " ← 未学習なので無い")

## ⑥ 書いてみる: `clone` + `set_params` でモデルの派生を作る

⑤ の表で、`Ridge(alpha=1.0)` と `Ridge(alpha=1000.0)` を**別々に書いて**並べた。
だが実務では「**基準のモデルを1つ決めて、そこから設定だけ変えた派生を作る**」書き方をしたい。
そうすれば基準を1か所直すだけで全派生に反映されるからだ(C# のプロトタイプパターン/オブジェクト初期化子の発想)。

次のセルで3つ作ろう。

| 変数 | 中身 |
|---|---|
| `ridge_weak` | `base_ridge` を**複製して** `alpha` を `1000.0` にしたモデル。**`base_ridge` 自体は変更しない** |
| `score_weak` | `ridge_weak` の CV RMSE(`cv_rmse` に渡すだけ) |
| `n_coef` | `ridge_weak` を **`X` 全体で `fit` したときの係数の本数**(`coef_` の要素数) |

使う道具: `clone(model)`(設定だけ複製)、`.set_params(alpha=...)`(自分自身を返す)、`.fit(X, y)`(自分自身を返す)、`len(...)`。
`coef_` は**学習後にしか存在しない**ので、`fit` してから触ること。

3行で書ける。`ridge_weak` を `cv_rmse` に渡すときは `fit` 済みである必要はない(中で `clone` して `fit` される)。

In [ ]:
base_ridge = Ridge(alpha=1.0)      # 出発点。これ自体は最後まで alpha=1.0 のままにしておく

ridge_weak = None   # ここに書く(ヒント: clone(...) してから .set_params(alpha=...)。両方とも自分自身を返す)
score_weak = None   # ここに書く(ヒント: cv_rmse(ridge_weak))
n_coef = None       # ここに書く(ヒント: clone(ridge_weak).fit(X, y).coef_ の長さ)

print("ridge_weak :", ridge_weak)
print("score_weak :", score_weak)
print("n_coef     :", n_coef)
print("base_ridge の alpha(壊していないか):", base_ridge.get_params()["alpha"])

In [ ]:
# ===== チェックポイント A: estimator API =====
check("A-1 ridge_weak の alpha", param_of(ridge_weak, "alpha"), 1000.0,
      hint="set_params(alpha=1000.0)。float で渡す(1000 でも数値として一致すれば OK)。")

check("A-2 base_ridge が壊れていないか", param_of(base_ridge, "alpha"), 1.0,
      hint="base_ridge.set_params(...) と直接書くと元が書き換わる。clone してから set_params する。")

check("A-3 ridge_weak の CV RMSE", score_weak, 0.959411,
      hint="0.637843 なら alpha が 1.0 のまま。cv_rmse(ridge_weak) の戻り値をそのまま入れる"
           "(return_oof=True にするとタプルが返るので注意)。")

check("A-4 係数の本数", n_coef, 17,
      hint="係数は特徴量1つにつき1本。X.shape[1] と一致するはず。coef_ は fit の後にだけ存在する。")

print("\n(4つとも [OK] になったら次の概念へ)")

---
# 概念2 — 木のアンサンブルはなぜテーブルに強いのか

## ① なぜ: 「とりあえず GBDT」が定石になっている理由を、自分で言えるようにする

テーブルコンペで上位陣がまず回すのは、ほぼ例外なく勾配ブースティング決定木(GBDT)だ。
実務でも「顧客の解約予測」「与信スコア」「需要予測」のような**表形式のデータ**では、
深層学習より GBDT の方が強いことが多い、というのが2020年代の共通認識になっている。

理由は3つある。**(1) 前処理がほとんど要らない**(標準化不要・外れ値に頑健・欠損をそのまま食べる)、
**(2) 特徴量どうしの掛け算(交互作用)を自動で拾う**、**(3) CPU で速い**。
この「前処理が要らない」は実務では単なる楽さではなく、**前処理リークの経路が減る**というリスク低減でもある。

だが「常に GBDT が勝つ」わけではない。**どんなデータなら木が強いのか**を、
今日は実測で確かめる。ここが分かっていないと、負けたときに何を疑えばいいか分からない。

## ② 解説: 1本の木 → 並列に束ねる → 直列に積む

### 決定木(Decision Tree)は「if の入れ子」

決定木は `if カテゴリ == 家電 then ... else if views > 120 then ...` という**分岐の入れ子**でしかない。
学習とは「どの列のどの閾値で切ると、目的変数のばらつきが一番減るか」を貪欲に選び続けることだ。

```
                 [category ∈ {家電}?]
                 /                  \
              yes                    no
        [brand_tier > 1?]        [condition == 新品?]
         /        \                 /         \
     9.9(葉)   10.4(葉)        8.6(葉)     8.1(葉)     ← 葉には「その領域の平均値」が入る
```

ここから**2つの性質**が出てくる。今日いちばん大事な性質だ。

- **単調変換に不変**。`views > 120` を `log1p(views) > log1p(120)` に書き換えても**同じ行が同じ側に落ちる**。
  だから **標準化・対数変換・単位の変更をしてもスコアが1ミリも変わらない**。外れ値も「一番端の領域」に入るだけなので影響が限定的。
- **外挿できない**。葉の値は学習データの平均なので、**学習時に見た範囲の外**の入力に対しては端の葉の値を返すだけ。
  時間トレンドのある列(このデータの `days`)では、未来を予測するときに効いてくる(unit04 で再訪)。

### 1本では弱い → 束ね方が2通りある

| | 決定木1本 | **ランダムフォレスト** | **勾配ブースティング(GBDT)** |
|---|---|---|---|
| 束ね方 | — | **並列**。少しずつ違うデータ・列で木を多数作り、**平均**する | **直列**。前の木の**残差(誤差)**を次の木が学び、**足し込む** |
| 何を減らす | — | **バリアンス**(1本の木が不安定なのを平均で均す) | **バイアス**(まだ説明できていない部分を順に潰す) |
| 木の深さ | — | 深い木(1本1本は過学習気味でよい) | 浅い木(1本1本は弱くてよい) |
| 木を増やすと | — | **過学習しにくい**(平均が安定するだけ) | **過学習する**(誤差を追いかけ続けるため) → だから早めに止める必要がある(= 概念3) |
| C# で言うと | 1つの実装 | 複数実装の結果を平均する `Aggregate` | デコレータを積み重ねて差分を補正していく |

```
ランダムフォレスト(並列)          勾配ブースティング(直列)
   木1 ─┐                          y - 予測 = 残差1
   木2 ─┼→ 平均 → 予測              木1 → 残差1 を学ぶ → 残差2
   木3 ─┘                                 木2 → 残差2 を学ぶ → 残差3 …
                                    予測 = 木1 + lr×木2 + lr×木3 + …
```

### 前処理が要るモデル / 要らないモデル

| モデル | 決定境界の作り方 | 標準化 | 欠損 |
|---|---|---|---|
| 決定木 / RF / GBDT | 軸に平行な**分割**(順序しか見ない) | **不要**(単調変換に不変) | LightGBM は**そのまま**扱える(欠損を専用の側に振り分ける) |
| 線形モデル(`Ridge` / `SGDRegressor`) | **重み付き和**。係数は列のスケールに反比例する | **必須級**。正則化は全係数に同じ強さで効くので、スケールがバラバラだと「単位の小さい列の係数」だけが不当に潰される | **不可**(`ValueError`)。埋めるしかない |
| 距離ベース(`kNN` / SVM) | **距離**。全列が同じ土俵に乗っている前提 | **必須**。値域の大きい列が距離を支配する | **不可** |

### 使う API

| 用途 | API | 主な引数 | 注意 |
|---|---|---|---|
| 決定木(回帰) | `DecisionTreeRegressor` | `max_depth`(深さ上限) | `max_depth=None` は「葉が純粋になるまで」= 全力で過学習 |
| ランダムフォレスト | `RandomForestRegressor` | `n_estimators`(木の本数) | 本数を増やしても過学習しない。増やすほど遅い |
| sklearn 版 GBDT | `HistGradientBoostingRegressor` | `max_iter`, `learning_rate` | LightGBM とほぼ同じ思想。欠損もそのまま扱える |
| 欠損の穴埋め | `SimpleImputer(strategy=...)` | `"mean" / "median" / "most_frequent"` | `fit` で埋める値を覚え、`transform` で適用する estimator |
| 標準化 | `StandardScaler` | — | 平均0・分散1にする。`fit` で平均と標準偏差を覚える |
| 前処理を繋ぐ | `make_pipeline(step1, step2, model)` | — | 全体が**1つの estimator** になる(`fit`/`predict` を持つ) |

> `SimpleImputer` や `StandardScaler` も estimator API に乗っている。違いは `predict` ではなく **`transform`** を持つこと。
> `make_pipeline` で繋ぐと「前段の `fit_transform` → 後段の `fit`」が自動で連鎖し、**全体がまた1つの estimator になる**。
> だから `cv_rmse(pipeline)` がそのまま動く。これは C# のデコレータ/ミドルウェア合成そのものだ。
> **パイプラインが「前処理を学習 fold だけで fit する」ことを構造的に保証してくれる**話は unit04 で主題として扱う。

In [ ]:
# GOAL: 木を1本 → 並列に束ねる → 直列に積む、で何が起きるかを CV スコアで見る

from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

# STEP 1: 決定木1本。深さを変えると「浅すぎ(未学習)」と「深すぎ(過学習)」の両端が見える
print("--- 決定木1本(max_depth を変える)---")
for d in [1, 3, 5, 8, None]:
    s = cv_rmse(DecisionTreeRegressor(max_depth=d, random_state=0))
    print(f"  max_depth={str(d):<5} CV RMSE={s:.6f}")
print("  ↑ 浅すぎても深すぎても悪化する。真ん中に最適がある(= バイアスとバリアンスの綱引き)")

# STEP 2: 束ねる。並列(RF)と直列(GBDT)
print("\n--- 束ねる ---")
print(f"  RandomForest(200本, 並列)      CV RMSE={cv_rmse(RandomForestRegressor(n_estimators=200, random_state=0, n_jobs=1)):.6f}")
print(f"  HistGradientBoosting(直列)     CV RMSE={cv_rmse(HistGradientBoostingRegressor(random_state=0)):.6f}")
print(f"  LightGBM(300本, 直列)          CV RMSE={cv_rmse(LGBMRegressor(**LGB_BASE)):.6f}")
print(f"  (参考)Ridge(alpha=1.0)         CV RMSE={cv_rmse(Ridge(alpha=1.0)):.6f}")
print("  ↑ 木を束ねると1本より良くなる。だが線形モデルには勝てていない")

# STEP 3: なぜ勝てないのか。このデータは「log を取ると足し算」だったから線形が有利。
#         逆に、木が圧勝するデータを作って見せる(x1 と x2 の "組み合わせ" でしか決まらない目的変数)
rng = np.random.default_rng(0)
n = 2000
x1, x2 = rng.random(n), rng.random(n)
y_toy = 2.0 * ((x1 > 0.5) ^ (x2 > 0.5)) + rng.normal(0, 0.2, n)   # ^ は排他的論理和(XOR)
X_toy = pd.DataFrame({"x1": x1, "x2": x2})
g_toy = pd.Series(np.arange(n))          # グループ構造の無いデータなので 1行 = 1グループ

print("\n--- 交互作用でしか決まらないデータ(X_toy:", X_toy.shape, ")---")
print(f"  Ridge(alpha=1.0)               CV RMSE={cv_rmse(Ridge(alpha=1.0), X=X_toy, y=y_toy, groups=g_toy):.6f}")
print(f"  DecisionTree(max_depth=4)      CV RMSE={cv_rmse(DecisionTreeRegressor(max_depth=4, random_state=0), X=X_toy, y=y_toy, groups=g_toy):.6f}")
print(f"  LightGBM(200本)                CV RMSE={cv_rmse(LGBMRegressor(n_estimators=200, learning_rate=0.1, random_state=0, n_jobs=1, verbose=-1), X=X_toy, y=y_toy, groups=g_toy):.6f}")
print("  ↑ 立場が逆転する。x1 単体・x2 単体では何も分からず『組み合わせ』でしか決まらないデータでは、")
print("    分割を積み重ねる木が圧勝する。線形モデルは x1, x2 の重み付き和しか作れないので手も足も出ない")

## ④ 予測: 単位を変える・欠損を入れる、と何が壊れる?

②で「木は単調変換に不変」「線形と距離ベースは前処理が必須」と書いた。本当か確かめる。

次のセルでは3つの実験をする。実行する前に予測しよう。

**実験1: `views` の単位を変える**(`views * 1000` にする / `log1p(views)` にする)

1. `DecisionTreeRegressor(max_depth=5)` の CV スコアはどうなる? **少し**変わる? **まったく**変わらない?
2. `KNeighborsRegressor(n_neighbors=5)`(最も近い5行の平均を返すモデル)はどうなる?

**実験2: 正則化を強めたときのスケールの影響**

3. `Ridge` の `alpha` を 1 → 10 → 100 → 1000 と上げていく。
   **生データ**と**標準化したデータ**で、スコアの差はどう開いていく?
4. `SGDRegressor`(確率的勾配降下法で線形モデルを解く)を**生データ**で回したらどうなる?

**実験3: `views` の12%を欠損(`NaN`)にする**

5. `Ridge` を欠損入りデータで `fit` したら? **エラーになる**? **無視して動く**?
6. `LGBMRegressor` は?

In [ ]:
# GOAL: 「木は前処理不要」「線形と距離ベースは前処理必須」を、同じデータで数値として確認する

from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# STEP 1: views の単位を変えた X を2つ作る(中身の順序は同じ、目盛りだけ違う)
X_x1000 = X.copy()
X_x1000["views"] = X_x1000["views"] * 1000        # 単位を変えただけ(順序は不変)
X_log = X.copy()
X_log["views"] = np.log1p(X_log["views"])         # 対数に潰しただけ(順序は不変)
print("X:", X.shape, " X_x1000:", X_x1000.shape, " X_log:", X_log.shape)
print("views の範囲  生:", (int(X["views"].min()), int(X["views"].max())),
      " ×1000:", (int(X_x1000["views"].min()), int(X_x1000["views"].max())),
      " log1p:", (round(float(X_log["views"].min()), 2), round(float(X_log["views"].max()), 2)))

print(f"\n{'モデル':<26}{'生':>11}{'views×1000':>13}{'log1p(views)':>14}")
print("-" * 64)
for name, make in [
    ("DecisionTree(depth=5)", lambda: DecisionTreeRegressor(max_depth=5, random_state=0)),
    ("LightGBM(300本)", lambda: LGBMRegressor(**LGB_BASE)),
    ("Ridge(alpha=10)", lambda: Ridge(alpha=10.0)),
    ("kNN(k=5)", lambda: KNeighborsRegressor(n_neighbors=5)),
]:
    print(f"{name:<22}{cv_rmse(make()):>11.6f}{cv_rmse(make(), X=X_x1000):>13.6f}{cv_rmse(make(), X=X_log):>14.6f}")
print("↑ 木は完全に同一の数字。距離ベースの kNN は動く。線形は alpha が小さいうちは影響が小さい")

# STEP 2: 正則化を強めると、スケールの差が牙をむく
print(f"\n{'モデル':<26}{'生データ':>12}{'標準化あり':>12}")
print("-" * 50)
for a in [1.0, 10.0, 100.0, 1000.0]:
    raw = cv_rmse(Ridge(alpha=a))
    scaled = cv_rmse(make_pipeline(StandardScaler(), Ridge(alpha=a)))
    print(f"{'Ridge(alpha=' + str(a) + ')':<22}{raw:>12.6f}{scaled:>12.6f}")
print(f"{'kNN(k=5)':<22}{cv_rmse(KNeighborsRegressor(n_neighbors=5)):>12.6f}"
      f"{cv_rmse(make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5))):>12.6f}")
sgd_raw = cv_rmse(SGDRegressor(random_state=0, max_iter=1000))
sgd_scaled = cv_rmse(make_pipeline(StandardScaler(), SGDRegressor(random_state=0, max_iter=1000)))
print(f"{'SGDRegressor':<22}{sgd_raw:>12.3e}{sgd_scaled:>12.6f}")
print("↑ kNN は定数予測(1.1119)より悪い = 完全に壊れている。SGD は勾配が発散して桁が飛んでいる")

# STEP 3: 欠損を入れる。views の12%を NaN にする(収集の失敗という設定)
rng_miss = np.random.default_rng(7)
X_miss = X.astype({"views": "float64"})
X_miss.loc[rng_miss.random(len(X_miss)) < 0.12, "views"] = np.nan
print("\nX_miss:", X_miss.shape, " views の欠損:", int(X_miss["views"].isna().sum()), "件")

try:
    cv_rmse(Ridge(alpha=10.0), X=X_miss)
except Exception as e:
    print("Ridge に欠損入りを渡すと →", type(e).__name__, ":", str(e).splitlines()[0])
print("LightGBM に欠損入りを渡すと → CV RMSE =", round(cv_rmse(LGBMRegressor(**LGB_BASE), X=X_miss), 6),
      "(欠損のない X では", round(cv_rmse(LGBMRegressor(**LGB_BASE)), 6), ")")

## ⑥ 書いてみる: 線形モデルを「前処理込みの1つの estimator」にする

⑤ で見た通り、`Ridge` は欠損入りデータでは `fit` すらできない。
そこで **欠損を埋める → 標準化する → Ridge** の3段を繋いで、**全体を1つの estimator** にする。
繋いだものは `fit` / `predict` を持つので、`cv_rmse` にそのまま渡せる。

次のセルで3つ作ろう。

| 変数 | 中身 |
|---|---|
| `pipe_ridge` | `SimpleImputer`(戦略は**中央値**)→ `StandardScaler` → `Ridge(alpha=10.0)` を繋いだもの |
| `score_pipe` | `pipe_ridge` の CV RMSE。**欠損入りの `X_miss`** で測る |
| `n_steps` | `pipe_ridge` の**段数**(何個の estimator が繋がっているか) |

使う道具:

- `make_pipeline(前処理1, 前処理2, モデル)` — 引数の順に繋ぐ。段の名前は自動で小文字クラス名になる
- `SimpleImputer(strategy="median")` — 各列の中央値で `NaN` を埋める
- `StandardScaler()` — 各列を平均0・分散1にする
- `cv_rmse(model, X=...)` — `X` はキーワード引数で差し替えられる(⑤ でやった通り)
- `pipe.steps` — `(名前, estimator)` のリスト

**なぜ imputer を先に置くのか**を考えてみよう(`StandardScaler` は `NaN` があると平均を計算できるだろうか?)。

In [ ]:
pipe_ridge = None   # ここに書く(ヒント: make_pipeline(SimpleImputer(strategy=...), StandardScaler(), Ridge(alpha=...)))
score_pipe = None   # ここに書く(ヒント: cv_rmse(pipe_ridge, X=X_miss))
n_steps = None      # ここに書く(ヒント: pipe_ridge.steps はリスト)

print("pipe_ridge :", pipe_ridge)
print("score_pipe :", score_pipe)
print("n_steps    :", n_steps)

In [ ]:
# ===== チェックポイント B: 前処理を繋いだ estimator =====
check("B-1 段数", n_steps, 3,
      hint="SimpleImputer → StandardScaler → Ridge の3段。len(pipe_ridge.steps) で数える。")

check("B-2 imputer の戦略", param_of(pipe_ridge, "simpleimputer__strategy"), "median",
      hint="make_pipeline が付ける段の名前は小文字クラス名。設定は 'simpleimputer__strategy' というキーで引ける"
           "(段名 + '__' + 引数名)。strategy='median' を指定する。")

check("B-3 Ridge の alpha", param_of(pipe_ridge, "ridge__alpha"), 10.0,
      hint="最後の段は Ridge(alpha=10.0)。")

check("B-4 欠損入りデータでの CV RMSE", score_pipe, 0.637585,
      hint="0.637779 なら欠損のない X で測っている(cv_rmse(pipe_ridge, X=X_miss) と渡す)。"
           "None のままなら pipe_ridge が作れていない。0.7078 なら LightGBM を渡している。")

print("\n(4つとも [OK] になったら次の概念へ)")

---
# 概念3 — LightGBM を回す: early stopping で「木の本数」を自動で決める

## ① なぜ: 「何本の木を積むか」は、人間が決められない

概念2 で見た通り、勾配ブースティングは**木を足すほど残差を追いかける**ので、
ある本数を超えると訓練データのノイズまで覚え始めて検証スコアが悪化する。
つまり `n_estimators`(木の本数)には**必ず最適値がある**。

そしてこの最適値は、データ・特徴量・`learning_rate`・**fold ごと**に変わる。
特徴量を1つ足しただけで最適本数は動く。だから「経験上300本くらい」と決め打ちするのは、
毎回わざと少し外れたところで止めているのと同じだ。

**early stopping** は、これを自動化する。「検証スコアが N 回連続で改善しなくなったら止める」だけの単純な仕組みだが、
これがあるおかげで `n_estimators` はチューニング対象から外れ、**残りのパラメータに集中できる**。
実務で GBDT を回すときの標準装備であり、**書き方が LightGBM のバージョンで変わった**ところなので、
ネットの記事をコピペすると必ず事故る箇所でもある。

## ② 解説: パラメータは「木の複雑さ × 学習の慎重さ × ランダム性」の3軸

### LGBMRegressor の主要パラメータ

| パラメータ | 何を制御するか | 増やすとどうなるか | 実務の目安 |
|---|---|---|---|
| `n_estimators` | **木の本数**(ブースティングの回数) | 表現力↑・過学習↑・学習時間↑ | early stopping 前提で大きめ(1000〜5000)に置く |
| `learning_rate` | 1本の木を足すときの**歩幅**(縮小率) | **小さくする**ほど1本の影響が減り、必要な本数は増え、最終スコアは上がりやすい | 0.01〜0.1。時間と相談 |
| `num_leaves` | 1本の木の**葉の最大数**(= 木の複雑さ) | 表現力↑・過学習↑ | 既定31。増やすなら `min_child_samples` も増やす |
| `max_depth` | 木の**深さ**の上限 | 表現力↑・過学習↑ | 既定 -1(無制限)。小さいデータでは 3〜8 で抑える |
| `min_child_samples` | 1つの葉に**最低何行**必要か | 増やすほど木が保守的に(過学習↓) | 既定20。数千行のデータでは増やす方向が効く |
| `subsample` + `subsample_freq` | 1本ごとに使う**行**の割合 | 減らすと多様性↑・過学習↓・速度↑ | 0.7〜1.0。**`subsample_freq>=1` にしないと効かない** |
| `colsample_bytree` | 1本ごとに使う**列**の割合 | 同上(列方向) | 0.7〜1.0 |
| `reg_lambda` | 葉の値への **L2 ペナルティ** | 予測が保守的に(過学習↓) | 0〜10 |
| `random_state` | 乱数 seed | — | **必ず固定**(再現性) |
| `n_jobs` | 使うスレッド数 | 大きいデータでは速くなる | 小さいデータでは `1` の方が速いことが多い |
| `verbose` | ログの量 | — | `-1` で黙る |

**過学習しているとき**に触る順番: `num_leaves`↓ → `min_child_samples`↑ → `colsample_bytree`/`subsample`↓ → `reg_lambda`↑。
**未学習のとき**はその逆 + `learning_rate`↓ & 本数↑。

### early stopping — LightGBM 4.x の作法

```python
model.fit(X_tr, y_tr,
          eval_set=[(X_va, y_va)],          # 監視する検証データ
          eval_metric="rmse",               # 監視する指標
          callbacks=[lgb.early_stopping(50, verbose=False),   # 50回改善しなければ停止
                     lgb.log_evaluation(0)])                  # 途中経過のログを黙らせる
model.best_iteration_      # 実際に採用された木の本数(= 検証スコアが最良だった時点)
model.best_score_          # そのときのスコア(dict の dict)
```

| 用途 | 書き方 | 戻り値・注意 |
|---|---|---|
| 検証データを渡す | `fit(..., eval_set=[(X_va, y_va)])` | **リスト**で複数渡せる。early stopping は**最後の**セットを見る |
| 監視指標 | `fit(..., eval_metric="rmse")` | 回帰の既定は `l2`(= MSE)。RMSE と最適点は同じだが表示が変わる |
| 早期終了 | `callbacks=[lgb.early_stopping(50, verbose=False)]` | 引数は「何回改善しなければ止めるか」(patience) |
| ログ抑制 | `callbacks=[lgb.log_evaluation(0)]` | 0 = 出力しない |
| 採用された本数 | `model.best_iteration_` | 学習後にだけ存在。early stopping が発動しなかった場合は `None` |
| 予測 | `model.predict(X)` | sklearn API では**自動的に `best_iteration_` までの木で**予測する |

> **重要(ネット記事の罠)** — **LightGBM 4.x では `fit()` に `early_stopping_rounds` 引数は存在しない。**
> 2〜3年前の記事にはほぼ全部 `model.fit(..., early_stopping_rounds=50)` と書いてあるが、
> 今これを実行すると学習が始まらずエラーになる。**`callbacks=[lgb.early_stopping(50)]` が唯一の正解**。
> 同様に `verbose=False` を `fit` に渡す書き方も廃止され、`lgb.log_evaluation(0)` に置き換わっている。

> **early stopping は検証 fold を見て本数を決めている。** つまり厳密には、その fold にほんの少しだけ適合している
> (CV スコアがわずかに楽観的になる)。Kaggle では標準的な運用として受け入れられているが、
> 「なぜ楽観的なのか」を説明できる状態にしておくこと。厳密にやるなら学習 fold をさらに分けて監視用を切り出す。

### カテゴリ列を LightGBM に直接食べさせる

概念1 では `get_dummies` で one-hot にした。LightGBM にはもう一つの道がある:
**pandas の `category` dtype にしておくと、LightGBM が自動的にカテゴリ特徴として扱う**
(one-hot に展開せず、「カテゴリの集合を2つに分ける」という分割を直接学習する)。

| 方法 | 列数 | 高カーディナリティ(数百種類)のとき |
|---|---|---|
| `pd.get_dummies` (one-hot) | カテゴリ値の数だけ増える | 列が爆発する。1列あたりの情報が薄くなる |
| `astype("category")` | 増えない | LightGBM が集合の分割を直接学ぶので有利になりやすい |

> **pandas 3.0 の罠** — 「文字列の列を探して `category` に変換する」ときに、
> ネットの記事は `df[c].dtype == object` で判定している。**pandas 3.0 では文字列列の dtype は `object` ではなく `str` なので、この判定は必ず False になる**(= 1列も変換されず、無言で素通りする最悪のバグ)。
> 正しくは **`pd.api.types.is_numeric_dtype(df[c])` の否定**を使う。この関数は「その列が数値型か」を返す。
> `isinstance(df[c].dtype, pd.CategoricalDtype)` で「もう category になっているか」も判定できる。

In [ ]:
# GOAL: early stopping で「fold ごとに最適な木の本数」が自動で決まる様子を見る

# STEP 1: n_estimators を大きく取り、止めるのは early stopping に任せる設定
LGB_ES = {**LGB_BASE, "n_estimators": 3000}
print("LGB_ES =", LGB_ES)


def cv_lgbm(params, Xd=X, y=y, groups=groups, es_rounds=50, verbose=False):
    """LightGBM を GroupKFold で回す。各 fold で early stopping を使い、
    (OOF の RMSE, fold ごとの best_iteration_ のリスト) を返す。"""
    gkf = GroupKFold(n_splits=5)
    oof = np.zeros(len(y))
    iters = []
    for k, (idx_tr, idx_va) in enumerate(gkf.split(Xd, y, groups)):
        X_tr, X_va = Xd.iloc[idx_tr], Xd.iloc[idx_va]
        y_tr, y_va = y[idx_tr], y[idx_va]
        model = LGBMRegressor(**params)
        callbacks = [lgb.log_evaluation(0)]
        if es_rounds:
            callbacks.insert(0, lgb.early_stopping(es_rounds, verbose=False))
        model.fit(X_tr, y_tr,
                  eval_set=[(X_va, y_va)],       # ← 監視する検証データ
                  eval_metric="rmse",            # ← 監視する指標
                  callbacks=callbacks)           # ← 4.x ではここに early stopping を書く
        oof[idx_va] = model.predict(X_va)        # 自動で best_iteration_ までの木を使う
        iters.append(model.best_iteration_)
        if verbose:
            fold_rmse = float(np.sqrt(np.mean((oof[idx_va] - y_va) ** 2)))
            print(f"  fold{k}: train={X_tr.shape} valid={X_va.shape} "
                  f"best_iteration_={model.best_iteration_:>5}  fold RMSE={fold_rmse:.6f}")
    return float(np.sqrt(np.mean((oof - y) ** 2))), iters


# STEP 2: 実行して fold ごとの停止位置を見る
print("\n--- LightGBM + early stopping(patience=50, n_estimators=3000 は上限)---")
score_es, iters_es = cv_lgbm(LGB_ES, verbose=True)
print(f"\nOOF RMSE = {score_es:.6f}   best_iteration_ = {iters_es}")
print(f"3000本まで許したのに、実際に使われたのは平均 {np.mean(iters_es):.1f} 本")
print("→ fold ごとに最適な本数は違う。学習データが変われば最適点も動く、というだけのこと")

# STEP 3: 概念1 で 300本固定にしたときのスコアと比べる
print(f"\n300本固定(概念1 の表): {cv_rmse(LGBMRegressor(**LGB_BASE)):.6f}")
print(f"early stopping あり  : {score_es:.6f}   ← 本数を自分で決めさせただけで改善した")

# STEP 4: 旧 API を使うとどうなるか(ネット記事のコピペで必ず踏む)
try:
    LGBMRegressor(**LGB_BASE).fit(X, y, eval_set=[(X, y)], early_stopping_rounds=50)
except TypeError as e:
    print("\nfit(..., early_stopping_rounds=50) を書くと →", type(e).__name__, ":", str(e).splitlines()[0])

## ④ 予測: 歩幅を変える / 早期終了をやめる

次のセルでは2つの比較をする。実行前に予測しよう。

**比較1: `learning_rate` を変える**(0.1 / 0.05 / 0.02)

1. `learning_rate` を**小さく**すると、`best_iteration_`(採用される木の本数)はどう動く?
2. CV スコアは良くなる? 悪くなる? どのくらい変わる?

**比較2: early stopping をやめて `n_estimators` を固定する**(50 / 200 / 1000 / 3000)

3. 本数を増やすほどスコアはどうなる? (ランダムフォレストなら「増やしても悪化しない」のだった)
4. 固定本数のベストは、early stopping ありのスコア(`0.669360`)に勝てる?

> `learning_rate` × `n_estimators` は**掛け算でひとつの量**になっている、という見方をすると予測しやすい。
> 歩幅を半分にしたら、同じ距離を進むには何歩必要だろうか。

In [ ]:
# GOAL: learning_rate と n_estimators の関係、および early stopping の有無を1枚の表で比べる

# STEP 1: learning_rate を変える(上限は 3000 本のまま、止めるのは early stopping)
print(f"{'learning_rate':<16}{'CV RMSE':>10}   {'fold ごとの best_iteration_'}")
print("-" * 66)
for lr in [0.1, 0.05, 0.02]:
    s, it = cv_lgbm({**LGB_ES, "learning_rate": lr})
    print(f"{lr:<16}{s:>10.6f}   {it}  (平均 {np.mean(it):.1f} 本)")
print("↑ 歩幅を半分にすると必要な本数はおよそ倍。掛け算でひとつの量になっている")

# STEP 2: early stopping をやめて本数を固定する(es_rounds=0 で無効化)
print(f"\n{'n_estimators(固定)':<20}{'CV RMSE':>10}")
print("-" * 32)
for ne in [50, 200, 1000, 3000]:
    s, _ = cv_lgbm({**LGB_BASE, "n_estimators": ne}, es_rounds=0)
    print(f"{ne:<20}{s:>10.6f}")
print(f"{'early stopping':<20}{score_es:>10.6f}   ← 本数を fold ごとに決めさせた場合")
print("↑ 木を増やすほど悪化する。これが『直列に残差を追いかける』ことの代償で、")
print("  ランダムフォレスト(並列・平均)との決定的な違い")

## ⑥ 書いてみる: カテゴリ列を `category` dtype にして LightGBM に渡す

ここまでは `get_dummies` で one-hot にした `X`(17列)を使ってきた。
今度は **one-hot にせず、カテゴリ列を `category` dtype のまま**渡す形を作る。LightGBM が自動で扱ってくれる。

次のセルで3つ作ろう。

| 変数 | 中身 |
|---|---|
| `X_cat` | `train[FEAT_NUM + FEAT_CAT]` のコピーで、**数値でない列だけ** `category` dtype にしたもの(7列) |
| `cat_cols` | 実際に `category` dtype になった列名の**リスト**(元の列の並び順のまま) |
| `score_cat`, `iters_cat` | `cv_lgbm(LGB_ES, X_cat)` の戻り値(タプルなので2つに展開できる) |

使う道具:

- `train[FEAT_NUM + FEAT_CAT].copy()` — リストの `+` は連結(C# の `Concat`)
- `pd.api.types.is_numeric_dtype(s)` — その Series が数値型かを返す。**`dtype == object` は pandas 3.0 では効かない**
- `s.astype("category")` — category dtype に変換する
- `isinstance(s.dtype, pd.CategoricalDtype)` — category dtype かどうかの判定
- `score, iters = cv_lgbm(params, Xd)` — タプルの展開代入

5〜7行で書ける。`for c in X_cat.columns:` で回して、数値**でない**列だけ変換するのが素直。

In [ ]:
X_cat = None       # ここに書く(ヒント: .copy() してから、数値でない列だけ astype("category"))
cat_cols = None    # ここに書く(ヒント: isinstance(X_cat[c].dtype, pd.CategoricalDtype) が True の列名)
score_cat = None   # ここに書く(ヒント: score_cat, iters_cat = cv_lgbm(LGB_ES, X_cat))
iters_cat = None

print("X_cat    :", None if X_cat is None else X_cat.shape)
print("cat_cols :", cat_cols)
print("score_cat:", score_cat, " iters_cat:", iters_cat)
if isinstance(X_cat, pd.DataFrame):
    print(X_cat.dtypes)

In [ ]:
# ===== チェックポイント C: category dtype と early stopping =====
check_frame("C-1 X_cat の形と列名", X_cat, shape=(1661, 7),
            columns=["brand_tier", "views", "title_len", "days", "category", "site", "condition"],
            hint="FEAT_NUM + FEAT_CAT の7列。列が17列なら get_dummies した X を使っている。"
                 "行数が違うなら train 以外を触っている。")

check("C-2 category dtype になった列", cat_cols, ["category", "site", "condition"],
      hint="空リストなら判定が効いていない(pandas 3.0 では dtype == object は必ず False)。"
           "pd.api.types.is_numeric_dtype(...) の否定で判定する。並び順は元の列順のまま。")

check("C-3 CV RMSE(小数第3位まで)",
      None if score_cat is None else round(float(score_cat), 3), 0.672,
      hint="0.669 なら one-hot の X を渡している(cv_lgbm(LGB_ES, X_cat) と渡す)。"
           "None のままなら戻り値のタプルを展開できていない。")

check("C-4 fold の数", None if iters_cat is None else len(iters_cat), 5,
      hint="cv_lgbm は fold ごとの best_iteration_ を5個のリストで返す。")

print("\n(4つとも [OK] になったら次の概念へ)")
print("※ この題材ではカテゴリが3〜5種類しかないので one-hot とほぼ同じ結果になる。")
print("  category dtype の真価は、数百種類あるカテゴリ列で効いてくる(unit04 で扱う)。")

---
# 概念4 — 特徴量重要度をどう読むか(そして、どう読み間違えるか)

## ① なぜ: 重要度は「次に何をするか」を決めるための道具

モデルが回ったあと、必ず聞かれることが2つある。「**何が効いているのか**」と「**次に何をすればいいのか**」だ。
特徴量重要度は、その両方に答えるための最初の道具になる。

- 効いていない列を削ると、学習が速くなり、過学習も減る(**特徴選択**)
- 効いている列の**周辺**に、次に作るべき特徴量のヒントがある(unit04 の特徴量エンジニアリングはここから始まる)
- そして最も重要な使い方 — **「本来使えないはずの列が上位に来ていないか」を確認する**。
  リーク列は必ず重要度トップに来る。**重要度表はリーク検出器**でもある

一方で、重要度は**読み間違えやすい**指標でもある。既定の計算方法には明確な癖があり、
「重要度が高い = 効いている」は言えても「重要度が高い = 原因である」は**言えない**。
実務でこれを取り違えると、「この施策を打てば売上が上がるはずです」という間違った提案になる。

## ② 解説: split と gain、そして「重要度は因果ではない」

### 2つの計算方法

| `importance_type` | 何を数えているか | 癖 |
|---|---|---|
| `"split"`(**既定**) | その列が**分割に使われた回数** | **高カーディナリティの列(値の種類が多い連続値)を過大評価する。** 切りどころが多いので何度でも使われるが、1回1回の効き目は小さい |
| `"gain"` | その分割によって**損失がどれだけ減ったか**の合計 | 普通はこちらの方が妥当。ただし相関の強い列どうしでは重要度が分散する |

```
split: 「何回登場したか」   → よく喋る人が偉い、という数え方
gain : 「どれだけ効いたか」 → 一言で場を変えた人が偉い、という数え方
```

| 用途 | API | 戻り値 |
|---|---|---|
| 学習済みモデルの重要度 | `model.feature_importances_` | `(n_features,)` の配列。**`importance_type` の設定に従う**(既定 `"split"`) |
| 種類を指定して取る | `model.booster_.feature_importance(importance_type="gain")` | 同上。**1つの学習済みモデルから両方取れる**ので学習し直さなくてよい |
| 列名 | `model.feature_name_` / 自分で持っている列リスト | 順序は学習時の列順 |

> `booster_` は「LightGBM の生のモデル本体」。`LGBMRegressor` は sklearn 互換の薄い皮であり、
> 中身の Booster にはこうして触れる。C# で言えばラッパクラスから内部実装を取り出す感覚。

### 重要度を読むときの3つの注意

1. **重要度は因果ではない。** 「アイスの売上」は「水難事故」の重要な予測子だが、原因は気温だ。
   重要度が高い列は「目的変数と一緒に動く」だけであり、**それを操作すれば結果が変わるとは限らない**。
2. **リーク列は必ずトップに来る。** 目的変数から作られた列は、当然どの本物の特徴量より強い。
   だから **重要度トップの列は、まずリークを疑ってから喜ぶ**。順番はこれ以外にない。
   チェックの型: 「その列は**予測したい時点で本当に手に入るか**?」「その列は目的変数から作られていないか?」
3. **相関の強い列があると重要度は分散する。** 同じ情報を持つ列が2本あると、木は気まぐれに片方を使うので、
   両方の重要度が半減して「どちらも大したことない」ように見える。

### より信頼できる方法: permutation importance

`sklearn.inspection.permutation_importance` は、**検証データ**の上で1列だけをシャッフルし、
スコアがどれだけ悪化したかを測る。「その列の情報を壊したら、実際どれだけ困るか」を直接測るので解釈がまっすぐで、
**モデルの種類を問わない**(線形でもニューラルネットでも使える)。
代わりに列数 × 繰り返し回数だけ予測を回すので重い。深入りはしないが、
「gain の順位が怪しいと思ったら permutation で裏を取る」という使い方だけ覚えておく。

In [ ]:
# GOAL: split と gain で重要度の順位が入れ替わることを見る

# 概念3 ⑦ で作った X_cat と同じものを、ここで教材側でも用意しておく(⑦ を飛ばしても先に進めるように)
X_tree = train[FEAT_NUM + FEAT_CAT].copy()
for c in FEAT_CAT:
    X_tree[c] = X_tree[c].astype("category")
print("X_tree:", X_tree.shape, " category 列:", list(X_tree.select_dtypes("category").columns))

# STEP 1: 重要度を見るためのモデルは「全データで1回 fit」する(CV の各 fold ではなく)
#   本数は early stopping で得た平均(概念3 の結果)くらいに置く
m_imp = LGBMRegressor(**{**LGB_BASE, "n_estimators": 200}).fit(X_tree, y)
print("学習後にだけ存在する属性:", "feature_importances_" , m_imp.feature_importances_.shape)

# STEP 2: 1つの学習済みモデルから split と gain の両方を取り出して並べる
imp = pd.DataFrame({
    "feature": list(X_tree.columns),
    "split": m_imp.booster_.feature_importance("split"),
    "gain": m_imp.booster_.feature_importance("gain").round(1),
})
print("\n--- gain の降順 ---")
print(imp.sort_values("gain", ascending=False).to_string(index=False))

print("\nsplit 順:", list(imp.sort_values("split", ascending=False)["feature"]))
print("gain  順:", list(imp.sort_values("gain", ascending=False)["feature"]))
print("\n→ 順位が入れ替わる。days / views / title_len は『連続値なので切りどころが無数にある』列で、")
print("  分割回数(split)は稼ぐが1回あたりの効き目は小さい。category は5種類しかないので")
print("  分割回数は少ないが、1回で損失を大きく減らしている。既定の split だけ見ると判断を誤る")

## ④ 予測: リーク列を混ぜると重要度はどうなる?

unit02 で「使ってはいけない」と確定した2列を、あえて特徴量に入れてみる。

| 列 | 素性 | test にあるか |
|---|---|---|
| `discount_rate` | `(list_price - price) / list_price` として作られた列。**目的変数から逆算されている** | **無い** |
| `list_price` | このデータでは `price` に 1.05〜1.9 倍を掛けて作られた列。やはり**目的変数から作られている** | **ある** |

実行する前に予測しよう。

1. CV RMSE(いまは `0.671646`)はどうなる? どのくらい下がる?
2. **gain** の重要度トップに来るのはどの列?
3. `list_price` は **test にも存在する**。「test にあるなら使ってよい」と言える? 言えないとしたらなぜ?
4. permutation importance(検証データで列をシャッフルしてスコアの劣化を測る)を、
   リーク列**なし**のモデルで取ったら、gain の順位と一致すると思う?

> 3 は今日いちばん大事な問いだ。**「本番でその列が手に入るか」と「その列が目的変数から作られていないか」は別の話**である。

In [ ]:
# GOAL: リーク列を入れると何が起きるかを、スコアと重要度の両方で見る

from sklearn.inspection import permutation_importance

# STEP 1: リーク列を足した特徴量行列を作る
X_leak = X_tree.copy()
X_leak["list_price"] = train["list_price"]
X_leak["discount_rate"] = train["discount_rate"]
print("X_tree:", X_tree.shape, "→ X_leak:", X_leak.shape)

# STEP 2: 同じ CV・同じモデルで、列を足しただけ
score_clean, _ = cv_lgbm(LGB_ES, X_tree)
score_leak, iters_leak = cv_lgbm(LGB_ES, X_leak)
print(f"\nリーク列なし CV RMSE = {score_clean:.6f}")
print(f"リーク列あり CV RMSE = {score_leak:.6f}   ← {score_clean / score_leak:.0f}倍以上『良く』なった")
print("スコアが跳ね上がったら喜ぶ前に疑う。unit02 で入れた規律がここで効く")

# STEP 3: 重要度を見るためのモデルを全データで学習しておく(⑦ で使う)
m_leak = LGBMRegressor(**{**LGB_BASE, "n_estimators": 200}).fit(X_leak, y)
print("\nm_leak を学習した。特徴量の数:", m_leak.n_features_, "/ 列名:", list(X_leak.columns))

# STEP 4: permutation importance(リーク列なしのモデルで)
#   1 fold ぶんだけ。検証データ上で1列をシャッフルし、スコアがどれだけ悪化したかを測る
idx_tr, idx_va = next(iter(GroupKFold(n_splits=5).split(X_tree, y, groups)))
m_perm = LGBMRegressor(**{**LGB_BASE, "n_estimators": 200}).fit(X_tree.iloc[idx_tr], y[idx_tr])
r = permutation_importance(m_perm, X_tree.iloc[idx_va], y[idx_va], n_repeats=5,
                           random_state=0, scoring="neg_root_mean_squared_error")
perm = pd.DataFrame({"feature": list(X_tree.columns), "劣化量": r.importances_mean.round(4)})
print("\n--- permutation importance(検証 fold 上、5回シャッフルの平均)---")
print(perm.sort_values("劣化量", ascending=False).to_string(index=False))
print("→ 『その列を壊すとスコアがどれだけ悪化するか』。gain の順位とだいたい合っているが、完全には一致しない")

## ⑥ 書いてみる: 重要度表を作る関数を書き、リークを検出する

重要度は毎回同じ形で見たいので、**関数にしておく**。
そしてそれを、⑤ で学習した**リーク列入りのモデル `m_leak`** に適用して、「何が起きているか」を読む。

次のセルで作ろう。

**1. `importance_table(model, feature_names)`** — 学習済みの LightGBM モデルから重要度表を作って返す関数

  - 返すのは列が `["feature", "split", "gain"]` の **DataFrame**(この順)
  - `gain` の**降順**に並べる
  - `reset_index(drop=True)` で行番号を 0 からの連番に振り直す(並べ替えると元の番号が残るため)
  - `gain` は `.round(1)` して見やすくする
  - 取り出し方は ③ の STEP 2 と同じ(`model.booster_.feature_importance("split") / ("gain")`)

**2. `imp_leak`** — `importance_table(m_leak, X_leak.columns)` の結果

**3. `top1`** — `imp_leak` の **gain 最上位の列名**(文字列。`imp_leak.loc[0, "feature"]` で取れる)

5〜8行で書ける。関数は「渡されたモデルの列数」に依存しないように、列名を引数で受け取ること。

In [ ]:
def importance_table(model, feature_names):
    """学習済み LightGBM モデルから ['feature','split','gain'] の表を作り、gain 降順で返す。"""
    # ここに書く(ヒント: pd.DataFrame({...}) を作って .sort_values("gain", ascending=False).reset_index(drop=True))
    return None


imp_leak = None   # ここに書く(ヒント: importance_table(m_leak, X_leak.columns))
top1 = None       # ここに書く(ヒント: gain 最上位の行の feature)

print(imp_leak)
print("gain トップ:", top1)

In [ ]:
# ===== チェックポイント D: 重要度表 =====
check_frame("D-1 imp_leak の形と列名", imp_leak, shape=(9, 3), columns=["feature", "split", "gain"],
            hint="X_leak は9列。列は ['feature','split','gain'] の順ちょうど。"
                 "pd.DataFrame({'feature': ..., 'split': ..., 'gain': ...}) で作れば順序はこの通りになる。")

check("D-2 gain トップの列名", top1, "list_price",
      hint="gain の降順に並べた1行目の feature。sort_values(..., ascending=False) の向きに注意。")

_names = col_of(imp_leak, "feature")
check("D-3 gain 上位3列", None if _names is None else _names[:3],
      ["list_price", "category", "discount_rate"],
      hint="9列すべてが入った表を gain 降順に並べる。reset_index(drop=True) を忘れると順序は合っていても loc が狂う。")

_splits = col_of(imp_leak, "split")
check("D-4 list_price が分割に使われた回数",
      None if (_names is None or "list_price" not in _names) else _splits[_names.index("list_price")], 2213,
      hint="split は整数(分割に使われた回数)。gain の配列を split 列に入れていないか確認。")

_clean = call_safely(importance_table, m_imp, X_tree.columns)
_clean_names = col_of(_clean, "feature")
check("D-5 関数がリーク列なしのモデルでも動くか(gain 上位3列)",
      None if _clean_names is None else _clean_names[:3], ["category", "days", "brand_tier"],
      hint="列名を引数で受け取り、モデルの列数に依存しない関数になっているか確認する。")

print("\n(5つとも [OK] になったら答え合わせへ)")
print("※ D-2 の結果が今日の結論そのもの: 重要度トップの列は、まずリークを疑ってから喜ぶ。")

---
## 答え合わせ: CV の順位は、本番(LB)でも保たれたか

CV は「本番のスコアを手元で当てるための道具」だ。だから最後に確認すべきは
**「CV で1位だったモデルが、本番でも1位だったか」**である(絶対値が一致することではない)。

今回は教材なので `test` の正解を見せる。次のセルは3つのモデルを**全データで再学習**して `test` を予測する:

| モデル | 全データ再学習のしかた |
|---|---|
| `DummyRegressor` | そのまま |
| `SimpleImputer + StandardScaler + Ridge` | そのまま(前処理も含めて `fit`) |
| `LightGBM` | **early stopping で得た `best_iteration_` の平均**を `n_estimators` にして再学習する |

最後の行が実務の型だ。**全データで学習するときは検証データが無いので early stopping が使えない。**
だから CV のときに各 fold が選んだ本数の平均を採用する(fold より学習データが増えるぶん、
少し多め(例: ×1.1)にする流儀もある)。

見るポイント:

1. **CV の順位と LB の順位が一致しているか**(絶対値のズレは気にしない)
2. early stopping ありと `n_estimators=3000` 固定で、**LB でも差がつくか**
3. `test` は `train` より**未来**の期間だということ(木は外挿できない、を思い出そう)

In [ ]:
# GOAL: 全データ再学習 → test を予測 → CV の順位が本番でも保たれたかを確認する

_UNIT_DIR = DATA.resolve().parent
ANSWER = _UNIT_DIR.parent / ".solutions" / _UNIT_DIR.name / "_answer.csv"
test = pd.read_csv(DATA / "test.csv", parse_dates=["collected_at"])
test["days"] = (test["collected_at"] - train["collected_at"].min()).dt.days   # 起点は train と同じ
print("train 期間:", train["collected_at"].min().date(), "〜", train["collected_at"].max().date())
print("test  期間:", test["collected_at"].min().date(), "〜", test["collected_at"].max().date(), " ← 未来")

if not ANSWER.exists():
    print("答えファイルが見つかりません(このセルはスキップして構いません):", ANSWER)
else:
    answer = pd.read_csv(ANSWER)
    y_test = np.log1p(answer.set_index("record_id").loc[test["record_id"], "price"].to_numpy(dtype=float))

    # STEP 1: test 側の特徴量を train と同じ形に揃える
    #   one-hot: reindex(columns=X.columns) で列の順序と有無を train に合わせる(unit01 の突き合わせ)
    X_test_oh = pd.get_dummies(test[FEAT_NUM + FEAT_CAT], columns=FEAT_CAT).reindex(columns=X.columns, fill_value=False)
    #   category: 水準(カテゴリの一覧)も train に合わせないと、内部のコード番号がズレる
    X_test_tree = test[FEAT_NUM + FEAT_CAT].copy()
    for c in FEAT_CAT:
        X_test_tree[c] = pd.Categorical(X_test_tree[c], categories=X_tree[c].cat.categories)
    print("\nX_test_oh:", X_test_oh.shape, " X_test_tree:", X_test_tree.shape)

    def rmse(a, b):
        return float(np.sqrt(np.mean((np.asarray(a, dtype=float) - np.asarray(b, dtype=float)) ** 2)))

    n_best = int(np.mean(iters_es))     # 概念3 の early stopping で得た本数の平均
    print("採用する木の本数(fold 平均):", n_best)

    rows = [
        ("DummyRegressor(mean)", 1.111857,
         DummyRegressor(strategy="mean").fit(X, y).predict(X_test_oh)),
        ("Imputer+Scaler+Ridge(10)", 0.637585,
         make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge(alpha=10.0)).fit(X, y).predict(X_test_oh)),
        (f"LightGBM(ES → {n_best}本)", score_clean,
         LGBMRegressor(**{**LGB_BASE, "n_estimators": n_best}).fit(X_tree, y).predict(X_test_tree)),
        ("LightGBM(3000本固定)", 0.754114,
         LGBMRegressor(**{**LGB_BASE, "n_estimators": 3000}).fit(X_tree, y).predict(X_test_tree)),
    ]
    print(f"\n{'モデル':<30}{'CV RMSE':>10}{'本番 LB':>10}")
    print("-" * 50)
    for name, cv, pred in rows:
        print(f"{name:<26}{cv:>10.6f}{rmse(y_test, pred):>10.6f}")
    print("\n※ CV と LB の絶対値はズレる。だが順位は保たれている ← これが『信じられる CV』ということ")
    print("※ early stopping ありと 3000本固定の差は、本番でもそのまま出ている")

---
## 振り返り(自己評価 + TIL)

以下に**1〜2文ずつ**、自分の言葉で書いてみよう。書いた内容はセッション終了時の学習ノートと
スキルレベルの判定に使う(空欄でも先に進めるが、言語化すると定着が大きく変わる)。

**1. 今日学んだことを自分の言葉で:**

> (ここに書く)

**2. 難しかったこと・まだあやふやなこと:**

> (ここに書く)

**3. Feynman チェック — 次の4つに、資料を見ずに答えられる?**

- 「CV ループのコードを変えずにモデルを差し替えられる」のはなぜか、C# の何に相当するか説明できる?
- 「木には標準化が要らないが `Ridge` には要る」のはなぜか、**決定境界の作り方**に触れて説明できる?
- LightGBM 4.x で early stopping を書くとき、`fit()` に何を渡す? 旧 API との違いは?
- 「`feature_importances_` の1位が `list_price` でした」と報告を受けたら、次に何を確認する?

> (ここに書く)

---
## まとめ

### 今日学んだこと

| # | 概念 | 一言でいうと |
|---|---|---|
| 1 | estimator API | 全モデルが `fit` / `predict` / `get_params` / `set_params` を実装している。**`IEstimator` を DI で差し替える構図そのもの** |
| 2 | `fit` は `self` を返す | だから `model.fit(X, y).predict(X)` と繋げられる |
| 3 | 末尾アンダースコア | `coef_` / `feature_importances_` / `best_iteration_` は**学習後にだけ存在する**属性 |
| 4 | `clone` | 設定だけ複製した**未学習**のモデルを作る。CV の各 fold は必ずまっさらから始める |
| 5 | 決定木 | 「if の入れ子」。**単調変換に不変**(標準化不要)、**外挿できない** |
| 6 | ランダムフォレスト | **並列**に多数の木を作って平均 → バリアンスを下げる。木を増やしても悪化しない |
| 7 | 勾配ブースティング | **直列**に残差を学んで足し込む → バイアスを下げる。**木を増やすと過学習する** |
| 8 | 木が強い場面 | 交互作用(組み合わせでしか決まらない関係)。逆に**加法的な構造なら線形モデルが強い**(今日の実測がまさにそれ) |
| 9 | 前処理の要否 | 木は不要。線形は標準化ほぼ必須+欠損不可(`ValueError`)。距離ベースは標準化必須(今日は kNN が定数予測より悪化した) |
| 10 | `make_pipeline` | 前処理器も estimator。繋ぐと**全体が1つの estimator** になり、そのまま CV に渡せる |
| 11 | early stopping | `n_estimators` を大きく取り、検証スコアが改善しなくなったら止める。**fold ごとに最適本数は違う** |
| 12 | LightGBM 4.x の作法 | `callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]`。**`fit(early_stopping_rounds=...)` は存在しない** |
| 13 | `category` dtype | LightGBM が自動でカテゴリ列として扱う。**pandas 3.0 では `dtype == object` 判定が効かない** → `pd.api.types.is_numeric_dtype` の否定 |
| 14 | split と gain | 既定の `split` は**高カーディナリティ列を過大評価する**。`gain` の方が普通は妥当。両方見る |
| 15 | 重要度の読み方 | **因果ではない**。相関列があると分散する。**リーク列は必ずトップに来る** |
| 16 | 全データ再学習 | early stopping で得た `best_iteration_` の平均を `n_estimators` にして学習し直す |

### 今日いちばん持ち帰るべき1行

**重要度トップの列は、まずリークを疑ってから喜ぶ。**

### この先どこで使うか(先読み)

- **unit04(特徴量エンジニアリング)** — 今日の主砲はそのまま。動かすのは**特徴量の側**になる。
  今日「`category` の gain が突出していた」という観察が、群ごとの集約特徴(`groupby`)を作る動機に直結する。
  `make_pipeline` は「前処理を学習 fold だけで fit することを構造的に保証する道具」として主題になる。
  高カーディナリティのカテゴリ列で `category` dtype と target encoding を比べるのもそこ。
- **unit05(テキスト)** — TF-IDF ベクトライザも `fit` / `transform` を持つ estimator。今日の型がそのまま効く。
- **unit06(名寄せ)** — ペアが同一実体かを当てる**二値分類**を LightGBM で解く。`predict` の代わりに `predict_proba` を使い、
  閾値を CV で決める。early stopping の作法は今日と同じ。
- **unit07 / unit09(深層)** — 「同じデータ・同じ CV で古典と深層を比べる」ときの**比較相手**が今日のモデルになる。
- **unit10(キャップストーン)** — 今日作った OOF 予測が**ブレンドの材料**になる。
  そして「木の本数」「モデルサイズ」は、そのまま推論コストの話になる。
- **実務** — 「表形式のデータで、まず何を回すか」への答えが今日の内容そのもの。
  定数 → 線形 → GBDT を同じ CV で並べ、重要度でリークを点検する。ここまでが1日目の仕事になる。

### 次にやること

**演習 `ex01_estimator_api` へ進もう。lesson.ipynb を見ながらで OK。**
思い出せない API があれば、②の表に戻ればいい。暗記ではなく、**どこを見れば分かるか**を覚えているのが実務の状態だ。

演習は4本:

| 演習 | 内容 |
|---|---|
| `ex01_estimator_api` | estimator を差し替えられる CV 関数を書く |
| `ex02_lgbm_early_stopping` | early stopping つきの LightGBM CV と `best_iteration_` の取り出し |
| `ex03_importance_vs_scaling` | 重要度の読み解きと、前処理が要る/要らないモデルの切り分け |
| `ex04_capstone` | 早期終了つき価格回帰と重要度監査を一気通貫 |